# Building and Deploying an MCP Server on Union

This tutorial walks you through building and deploying a **Model Context Protocol (MCP)** server on Union that helps AI agents find recipes! You'll create a recipe assistant that can search by ingredients, dietary needs, and more using the [Spoonacular Food API](https://spoonacular.com/food-api).

### What You'll Learn

1. **What MCP is** and how it enables AI agents to connect with external tools
2. **Setting up the Spoonacular API** (just one API key!)
3. **Building MCP tools** with Python using the official MCP Python SDK
4. **Testing locally** before deployment
5. **Deploying to Union** for production use
6. **Connecting to AI agents** in Cursor or Claude Code

---

## Part 1: Understanding MCP

### What is the Model Context Protocol?

The [Model Context Protocol](https://modelcontextprotocol.io/) (MCP) is an open standard that enables AI assistants to securely connect with external data sources and tools. Think of it as a universal adapter that lets AI agents interact with APIs, databases, and services.

```
┌─────────────────┐     MCP Protocol     ┌─────────────────┐     API     ┌─────────────────┐
│   AI Agent      │◄────────────────────►│   MCP Server    │◄───────────►│   Spoonacular   │
│ (Claude, etc.)  │    Tools & Resources │ (Your Server)   │             │    Food API     │
└─────────────────┘                      └─────────────────┘             └─────────────────┘
```

### Why a Recipe Assistant?

Everyone eats! A recipe assistant is:
- **Universally relatable** - helps with daily meal planning
- **Practical** - real value for users
- **Great for demos** - visual, engaging results

---

## Part 2: Get Your API Key (2 minutes!)

### Setting Up Spoonacular

1. Go to [spoonacular.com/food-api](https://spoonacular.com/food-api)
2. Click **Start Now** and create a free account
3. Copy your API key from the dashboard

**That's it!** The free tier includes **150 points/day** - plenty for development and testing!

### API Key Safety

Never commit your API key to git! We'll use environment variables.

Create a `.env` file in this folder:

```bash
SPOONACULAR_API_KEY=your-api-key-here
```

In [ ]:
# Set your API key here (or use a .env file)
import os

# Option 1: Set directly (for testing only - don't commit this!)
# os.environ["SPOONACULAR_API_KEY"] = "your-api-key-here"

# Option 2: Load from .env file (recommended)
from dotenv import load_dotenv
load_dotenv()

# Check if API key is set
api_key = os.getenv("SPOONACULAR_API_KEY")
if api_key:
    print(f"✅ API key loaded! (ends with ...{api_key[-4:]})")
else:
    print("❌ No API key found. Set SPOONACULAR_API_KEY environment variable.")

### Project Structure

```
tutorials/mcp/
├── README.md                           # Documentation
├── requirements.txt                    # Python dependencies
├── server.py                           # MCP server implementation
├── tools/                              # Spoonacular API tools
│   ├── __init__.py
│   └── recipes.py                      # Recipe API wrapper
├── app.py                              # Union app deployment script
└── tutorial_recipe_mcp.ipynb           # This notebook
```

---

## Part 3: Building the Recipe Client

Let's create a simple client to interact with the Spoonacular API. This will be the foundation for our MCP tools.

In [ ]:
import httpx

class RecipeClient:
    """Simple client for Spoonacular Food API."""
    
    BASE_URL = "https://api.spoonacular.com"
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.client = httpx.Client(timeout=30.0)
    
    def _request(self, endpoint: str, params: dict = None) -> dict:
        """Make API request."""
        params = params or {}
        params["apiKey"] = self.api_key
        response = self.client.get(f"{self.BASE_URL}{endpoint}", params=params)
        response.raise_for_status()
        return response.json()
    
    def search_recipes(self, query: str, number: int = 5) -> list:
        """Search for recipes by name/description."""
        result = self._request("/recipes/complexSearch", {"query": query, "number": number})
        return result.get("results", [])
    
    def search_by_ingredients(self, ingredients: list[str], number: int = 5) -> list:
        """Find recipes using ingredients you have."""
        return self._request("/recipes/findByIngredients", {
            "ingredients": ",".join(ingredients),
            "number": number,
            "ranking": 1,  # Maximize used ingredients
        })
    
    def get_recipe_info(self, recipe_id: int) -> dict:
        """Get detailed recipe information."""
        return self._request(f"/recipes/{recipe_id}/information")

# Create client
client = RecipeClient(api_key)
print("✅ Recipe client ready!")

In [ ]:
# 🧪 Test: Search for pasta recipes
recipes = client.search_recipes("pasta carbonara", number=3)

print("🍝 Found recipes:")
for r in recipes:
    print(f"  - {r['title']} (ID: {r['id']})")

In [ ]:
# 🧪 Test: "What's in my fridge?" search
my_ingredients = ["chicken", "rice", "garlic", "onion"]
recipes = client.search_by_ingredients(my_ingredients, number=3)

print(f"🍗 Recipes using {my_ingredients}:")
for r in recipes:
    used = [i["name"] for i in r.get("usedIngredients", [])]
    missing = [i["name"] for i in r.get("missedIngredients", [])]
    print(f"\n  📖 {r['title']}")
    print(f"     ✅ Uses: {', '.join(used)}")
    print(f"     🛒 Need: {', '.join(missing) if missing else 'Nothing else!'}")

---

## Part 4: Creating MCP Tools

Now let's wrap our recipe functionality into MCP tools using the official MCP Python SDK.

In [ ]:
from mcp.server.fastmcp import FastMCP

# Initialize the MCP server
mcp = FastMCP(
    name="recipe-assistant",
    instructions="""
    You are a helpful recipe assistant. You can:
    - Search for recipes by name or description
    - Find recipes using ingredients the user has
    - Get detailed recipe information
    
    Always be helpful and suggest alternatives when appropriate!
    """
)

print("✅ MCP server initialized!")

In [ ]:
# Define MCP tools using decorators

@mcp.tool()
async def search_recipes(
    query: str,
    cuisine: str = None,
    diet: str = None,
    number: int = 5,
) -> list[dict]:
    """
    Search for recipes by name, cuisine, or dietary preference.
    
    Args:
        query: What to search for (e.g., "pasta", "quick dinner", "chocolate cake")
        cuisine: Optional cuisine filter (italian, mexican, asian, etc.)
        diet: Optional diet filter (vegan, vegetarian, gluten free, keto, etc.)
        number: How many recipes to return (default 5)
    
    Returns:
        List of matching recipes with titles and IDs
    """
    params = {"query": query, "number": number}
    if cuisine:
        params["cuisine"] = cuisine
    if diet:
        params["diet"] = diet
    
    result = client._request("/recipes/complexSearch", params)
    return result.get("results", [])


@mcp.tool()
async def search_by_ingredients(
    ingredients: list[str],
    number: int = 5,
) -> list[dict]:
    """
    Find recipes using ingredients you have on hand.
    
    This is the "what's in my fridge" tool - perfect for reducing food waste!
    
    Args:
        ingredients: List of ingredients you have (e.g., ["chicken", "rice", "garlic"])
        number: How many recipes to return (default 5)
    
    Returns:
        Recipes showing which ingredients are used and which you'd need to buy
    """
    recipes = client.search_by_ingredients(ingredients, number)
    
    # Format for clarity
    return [
        {
            "id": r["id"],
            "title": r["title"],
            "used_ingredients": [i["name"] for i in r.get("usedIngredients", [])],
            "missing_ingredients": [i["name"] for i in r.get("missedIngredients", [])],
        }
        for r in recipes
    ]


@mcp.tool()
async def get_recipe_details(recipe_id: int) -> dict:
    """
    Get full details for a specific recipe including ingredients and instructions.
    
    Args:
        recipe_id: The recipe ID (from a previous search)
    
    Returns:
        Complete recipe with ingredients, instructions, and nutrition info
    """
    recipe = client.get_recipe_info(recipe_id)
    
    return {
        "title": recipe.get("title"),
        "servings": recipe.get("servings"),
        "ready_in_minutes": recipe.get("readyInMinutes"),
        "ingredients": [
            f"{i['amount']} {i['unit']} {i['name']}"
            for i in recipe.get("extendedIngredients", [])
        ],
        "instructions": recipe.get("instructions", "No instructions available"),
        "diets": recipe.get("diets", []),
    }


print("✅ MCP tools defined!")
print("\nRegistered tools:")
print("  - search_recipes")
print("  - search_by_ingredients") 
print("  - get_recipe_details")

---

## Part 5: Testing the MCP Tools

Let's test our tools before deploying!

In [ ]:
# Test: Search for vegan Italian recipes
print("🧪 Testing search_recipes...")
results = await search_recipes("pasta", cuisine="italian", diet="vegan", number=3)
for r in results:
    print(f"  🍝 {r['title']}")

In [ ]:
# Test: What can I make with these ingredients?
print("\n🧪 Testing search_by_ingredients...")
results = await search_by_ingredients(["salmon", "lemon", "dill"], number=3)
for r in results:
    print(f"\n  🐟 {r['title']}")
    print(f"     ✅ Uses: {', '.join(r['used_ingredients'])}")
    print(f"     🛒 Need: {', '.join(r['missing_ingredients']) or 'Nothing!'}")

In [ ]:
# Test: Get full recipe details
print("\n🧪 Testing get_recipe_details...")
if results:
    recipe_id = results[0]["id"]
    details = await get_recipe_details(recipe_id)
    
    print(f"\n📖 {details['title']}")
    print(f"   ⏱️  Ready in {details['ready_in_minutes']} minutes")
    print(f"   🍽️  Serves {details['servings']}")
    print(f"\n   Ingredients:")
    for ing in details['ingredients'][:5]:  # First 5
        print(f"     • {ing}")
    if len(details['ingredients']) > 5:
        print(f"     ... and {len(details['ingredients']) - 5} more")

---

## Part 6: Running the MCP Server Locally

### Start the Server

```bash
python server.py
```

### Test with MCP Inspector

```bash
npx -y @modelcontextprotocol/inspector
```

In the inspector UI, connect to the server at http://localhost:8000/mcp

### Connect to Claude Code

```bash
claude mcp add --transport http spoonacular-mcp http://localhost:8000/mcp
```

### Example Queries

Once connected to an AI agent, you can ask things like:

- *"What can I make with chicken, rice, and broccoli?"*
- *"Find me a vegan pasta recipe under 500 calories"*
- *"I want something similar to beef stroganoff"*
- *"Show me high-protein breakfast ideas"*
- *"What's a good gluten-free dessert?"*

### Cleanup

When done testing locally, remove the MCP server from Claude:

```bash
claude mcp remove spoonacular-mcp
```

---

## Part 7: Deploying to Union

Deploy your MCP server to Union for production use!

### 1. Connect to Union

```bash
# Configure Union CLI
flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --auth-type headless \
    --builder remote \
    --domain development \
    --project workshops

# Store your API key as a secret
flyte create secret SPOONACULAR_API_KEY
```

### 2. Deploy the MCP Server

Set a name for your app:

```bash
export APP_NAME=<my-app-name>
```

Build and deploy the container image:

```bash
python app.py
```

### 3. Configure Your MCP Client

**For Claude Code:**

```bash
claude mcp add --transport http spoonacular-mcp <app_url>/spoonacular/mcp
```

Where `<app_url>` looks something like: `https://<subdomain>.tryv2.hosted.unionai.cloud`

**For Cursor** (`~/.cursor/mcp.json`):

```json
{
  "mcpServers": {
    "recipe-assistant": {
      "url": "https://<subdomain>.apps.tryv2.hosted.unionai.cloud/spoonacular/mcp"
    }
  }
}
```

Test it by asking: "What can I make with chicken and rice?"

---

## Part 8: Securing the MCP Server

To secure the MCP server, you can use the `REQUIRES_AUTH` environment variable in `app.py`.

### Redeploy with Authentication

```bash
python app.py
```

Now connections will fail without proper authentication.

### Create a Flyte API Key

```bash
flyte create api-key --name <api-key-name>
```

Save the `<FLYTE_API_KEY>` string somewhere safe!

### Re-configure MCP Connection

**For Claude Code:**

```bash
claude mcp remove spoonacular-mcp
claude mcp add --transport http spoonacular-mcp <app_url>/spoonacular/mcp --header "Authorization: Bearer <FLYTE_API_KEY>"
```

**For Cursor** (`~/.cursor/mcp.json`):

```json
{
  "mcpServers": {
    "recipe-assistant": {
      "url": "https://<subdomain>.apps.tryv2.hosted.unionai.cloud/spoonacular/mcp",
      "headers": {
        "Authorization": "Bearer <FLYTE_API_KEY>"
      }
    }
  }
}
```

### Rotating API Keys

```bash
flyte delete api-key <api-key-name>
flyte create api-key --name <api-key-name>
```

---

## Summary

You built a Recipe Assistant MCP server that can:

- Search recipes by name, cuisine, and dietary needs
- Find recipes using ingredients you have ("what's in my fridge")
- Search by nutritional requirements
- Get detailed recipe information with ingredients and instructions
- Find similar recipes to ones you like
- Autocomplete recipe names

### Available Tools

| Tool | Description |
|------|-------------|
| `search_recipes` | Search recipes by name, cuisine, diet, and more |
| `search_by_ingredients` | Find recipes using ingredients you have |
| `search_by_nutrients` | Find recipes by nutritional requirements |
| `get_recipe_info` | Get detailed recipe information |
| `get_similar_recipes` | Find recipes similar to one you like |
| `autocomplete_recipe` | Get recipe name suggestions |

### Next Steps

- Add more tools (meal planning, wine pairing, nutrition analysis)
- Build a recipe recommendation workflow
- Integrate with shopping list APIs
- Create a meal prep planning agent

### Resources

- [Model Context Protocol Specification](https://modelcontextprotocol.io/)
- [Union MCP Reference Implementation](https://github.com/unionai-oss/union-mcp)
- [Spoonacular API Documentation](https://spoonacular.com/food-api/docs)
- [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk)